# Báo cáo Khoa học: Phân loại Bệnh Cây trồng với Kiến trúc MobileNetV3-small
**Mục tiêu:** Xây dựng pipeline học máy chuẩn doanh nghiệp tối ưu cho Edge Devices dựa trên nghiên cứu của Khan (2023) trên tập PlantVillage (54,309 ảnh, 38 lớp).

## 1. Cơ sở Toán học Tối ưu hóa (Edge Friendly)
Kiến trúc MobileNetV3 sử dụng các hàm kích hoạt Hard-Swish và Hard-Sigmoid để tránh việc tính toán hàm số mũ phức tạp trên thiết bị phần cứng thấp:
$$ \text{h-sigmoid}(x) = \frac{\text{ReLU6}(x + 3)}{6} $$
$$ \text{h-swish}(x) = x \cdot \text{h-sigmoid}(x) $$

Khối **Squeeze-and-Excite (SE)** học trọng số tương quan kênh (Channel Attention):
$$ S = \text{h-sigmoid}(W_2 \cdot \text{ReLU}(W_1 \cdot Z)) $$

## 2. Kiến trúc Bottleneck & Tensor Flow
Khởi chạy với ảnh đầu vào `(1, 3, 224, 224)`, đi qua 11 khối Inverted Residual Bottlenecks để trích xuất feature maps, và cuối cùng phân loại ra 38 classes bằng lớp Dense layer thông qua thiết kế `nn.Linear`.


In [ ]:
# SETUP MÔI TRƯỜNG KAGGLE & REPRODUCIBILITY SEEDS
import os
import sys
import random
import numpy as np
import torch
import warnings

warnings.filterwarnings('ignore')

# Thiết lập đường dẫn module src/ (Đảm bảo upload folder source code lên Kaggle /kaggle/working)
project_dir = '/kaggle/working/project_ck_nhom01'
if project_dir not in sys.path:
    sys.path.append(project_dir)

def seed_everything(seed=42):
    """Đảm bảo tính tái lập (reproducibility) 100% cho các thí nghiệm"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
print("Environment Setup Complete. Seeds locked tightly for academic reproducibility.")


In [ ]:
# KHẢO SÁT DỮ LIỆU (EDA), BẢNG PHÂN PHỐI & VISUALIZATION SAMPLES
import yaml
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from torchvision.utils import make_grid
from torch.utils.data import DataLoader
from src.data.dataset import PlantVillageDataset
from src.data.transforms import get_baseline_transforms

with open('configs/default_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

dataset_dir = config['data']['dataset_dir']

if os.path.exists(dataset_dir):
    dataset = PlantVillageDataset(dataset_dir, transform=get_baseline_transforms(224))
    class_counts = Counter(dataset.labels)
    
    # 1. In bảng phân phối (Table 1)
    distribution_data = {
        "Class Index": list(class_counts.keys()),
        "Class Name": [dataset.classes[idx] for idx in class_counts.keys()],
        "Number of Images": list(class_counts.values())
    }
    df_dist = pd.DataFrame(distribution_data).sort_values("Number of Images", ascending=False)
    
    print("\n=== Bảng 1: Phân phối mất cân bằng (Imbalanced) của PlantVillage ===")
    display(df_dist.style.background_gradient(cmap='Blues'))
    print(f"\nTổng số ảnh phát hiện: {sum(class_counts.values())} across {len(class_counts)} classes.\n")
    
    # 2. Render lưới ảnh mẫu (2x4 Grid) để xác thực Pipeline biến đổi dữ liệu
    loader = DataLoader(dataset, batch_size=8, shuffle=True)
    images, labels = next(iter(loader))
    
    # Hàm Unnormalize
    def imshow(img, title):
        img = img.numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        plt.imshow(img)
        plt.title(title, fontsize=10)
        plt.axis('off')
        
    plt.figure(figsize=(16, 8))
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        imshow(images[i], title=dataset.classes[labels[i].item()])
    plt.suptitle("Sample Preprocessed Images (Unnormalized)", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print(f"WARNING: Data Directory '{dataset_dir}' not found. Attach Dataset on Kaggle.")


In [ ]:
# ĐỊNH NGHĨA MODULAR MODULES & ARCHITECTURE SANITY CHECK
from src.models.mobilenet_v3 import MobileNetV3Small
from torchinfo import summary

# Khởi tạo mô hình tự xây dựng từ Scratch
model = MobileNetV3Small(num_classes=config['model']['num_classes'])

# Xác thực số lượng tham số huấn luyện (~1.54M Params theo Paper Baseline)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Tổng số tham số học được (Trainable Parameters): {total_params:,}")

# In cấu trúc bảng kiến trúc tổng quan (Sanity check hình khối Tensor)
print("\n=== Bảng 2: Cấu trúc Tensor Flow (Replicating Table 2) ===")
summary(model, input_size=(1, 3, 224, 224))


In [ ]:
# PHẪU THUẬT TRỌNG SỐ TỪ IMAGENET (State_Dict Surgery)
from src.utils.weight_loader import migrate_imagenet_weights

print("Khởi chạy quy trình Phẫu thuật Trọng số (State_Dict Surgery)...")
try:
    # Strict=True kiểm chứng cấu trúc cell-by-cell match 100%
    model = migrate_imagenet_weights(model, num_classes=config['model']['num_classes'])
    print("SUCCESS: Kiến trúc đã vượt qua Validation `strict=True`. Trọng số ImageNet nạp thành công!")
except Exception as e:
    print(f"FAILED: Mismatch Architecture. {e}")


In [ ]:
# OPTUNA HYPERPARAMETER TUNING (Không gian tìm kiếm Augmentation/HPO)
from src.tune import tune_hyperparameters

print("Khởi chạy HPO (Tìm kiếm Siêu tham số với Optuna)...")
# Ghi chú: Chạy để chứng minh sự cải tiến.
# tune_hyperparameters() 

print("Tuning Module là thiết kế tích hợp. Cấu hình tốt nhất được quy hoạch vào yaml config.")


In [ ]:
# VÒNG LẶP HUẤN LUYỆN CHÍNH (Training Engine)
from src.train import train_model

print("Khởi động Training Loop...")
print("Cơ chế Checkpoint: Lưu tệp trọng số khi Validation Macro F1-Score đạt đỉnh mới.")

# Khởi chạy hàm train_model. (Đảm bảo train_model trả về history_dict chứa mảng loss và metric)
history = train_model(config_path="configs/default_config.yaml")

print("Training Completed. Lịch sử Loss và Metric đã được thu thập cho Cell 8.")


In [ ]:
# ĐÁNH GIÁ TRỰC QUAN & KHÔNG GIAN ĐẶC TRƯNG (Replicating Figures 5, 6, 7 & Table 3)
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from src.utils.visualization import extract_features_and_tsne
from src.models.mobilenet_v3 import MobileNetV3Small
from torch.utils.data import DataLoader
from IPython.display import Image, display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.path.exists('checkpoints/best_model.pth') and 'history' in locals() and history is not None:
    # --- A. BIỂU ĐỒ HỘI TỤ (Learning Curves) ---
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Figure 5: Epoch-to-Loss
    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='blue', marker='o')
    axes[0].plot(epochs, history['val_loss'], label='Val Loss', color='orange', marker='s')
    axes[0].set_title('Figure 5: Training & Validation Loss', fontsize=14)
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # Figure 6: Epoch-to-Accuracy
    axes[1].plot(epochs, history['train_acc'], label='Train Accuracy', color='green', marker='o')
    axes[1].plot(epochs, history['val_acc'], label='Val Accuracy', color='red', marker='s')
    axes[1].set_title('Figure 6: Training & Validation Accuracy', fontsize=14)
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

    # --- B. ĐÁNH GIÁ MÔ HÌNH VỚI BEST CHECKPOINT ---
    best_model = MobileNetV3Small(num_classes=config['model']['num_classes'])
    best_model.load_state_dict(torch.load('checkpoints/best_model.pth', map_location=device))
    best_model = best_model.to(device)
    best_model.eval()

    # Tái tạo Validation Set cho Evaluation
    val_dataset_full = PlantVillageDataset(dataset_dir, transform=get_baseline_transforms(224))
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(len(val_dataset_full), generator=generator).tolist()
    train_size = int(0.8 * len(val_dataset_full))
    val_dataset = torch.utils.data.Subset(val_dataset_full, indices[train_size:])
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    all_preds = []
    all_labels = []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            outputs = best_model(imgs.to(device))
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(lbls.numpy())

    # --- C. CLASSIFICATION REPORT (Replicating Table 3) ---
    target_names = val_dataset_full.classes
    print("\n=== Bảng 3: Chi tiết Classification Report (Precision, Recall, F1) ===")
    print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))

    # --- D. CONFUSION MATRIX HEATMAP ---
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, annot=False, cmap='Blues', fmt='g', xticklabels=target_names, yticklabels=target_names)
    plt.title('Complete 38x38 Confusion Matrix Heatmap', fontsize=16)
    plt.xlabel('Predicted Class', fontsize=12)
    plt.ylabel('True Class', fontsize=12)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # --- E. t-SNE VISUALIZATION (Figure 7) ---
    # Lấy ngẫu nhiên 1000 mẫu để vẽ t-SNE nhằm tiết kiệm RAM trên Kaggle
    tsne_subset_indices = torch.randperm(len(val_dataset_full))[:1000]
    tsne_subset = torch.utils.data.Subset(val_dataset_full, tsne_subset_indices)
    tsne_loader = DataLoader(tsne_subset, batch_size=64, shuffle=False)
    
    # Hàm extract_features_and_tsne sẽ render trực tiếp nếu loại bỏ plt.close() hoặc lưu ra file
    extract_features_and_tsne(best_model, tsne_loader, device, save_path="tsne_figure7.png")

    print("\n=== Figure 7: T-SNE 1024-D Latent Space Projection ===")
    display(Image("tsne_figure7.png"))
else:
    print("Mô hình huấn luyện chưa có sẵn hoặc thiếu lịch sử loss. Hãy chạy Cell 7 trước.")


In [ ]:
# EDGE COMPUTING EXPORT & DYNAMIC QUANTIZATION RATIO
import os
from src.export import export_model_for_edge

print("=== CHUẨN BỊ TRIỂN KHAI THIẾT BỊ BIÊN (EDGE EXPORT) ===")
export_model_for_edge(config_path="configs/default_config.yaml")

# Explicit Compression Ratio Calculation
orig_path = "checkpoints/best_model.pth"
quant_path = "checkpoints/best_model_quantized.pt"

if os.path.exists(orig_path) and os.path.exists(quant_path):
    orig_size = os.path.getsize(orig_path) / (1024 * 1024)
    quant_size = os.path.getsize(quant_path) / (1024 * 1024)
    
    print("\n=== MINH CHỨNG TỐI ƯU HÓA THIẾT BỊ BIÊN (EDGE OPTIMIZATION) ===")
    print(f"Original Checkpoint Size (.pth): {orig_size:.2f} MB")
    print(f"Quantized TorchScript Size (.pt): {quant_size:.2f} MB")
    print(f"Compression Ratio: {orig_size / quant_size:.2f}x smaller")
    print("Mô hình đã sẵn sàng để nhúng vào nền tảng di động Android (Opset 17 ONNX hoặc Quantized PT).")
